# FlowOS V2: Remote SFT -> GRPO on Hugging Face Jobs

This notebook launches remote HF Jobs instead of training on Kaggle GPU.

Pipeline:
1. Collect ranked traces remotely
2. Train SFT remotely on `l4x1`
3. Upload the SFT checkpoint to HF Hub
4. Launch GRPO from that SFT checkpoint
5. Monitor or cancel jobs from this notebook

In [9]:
!pip install -q -U huggingface_hub

In [10]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("HF auth ready")

HF auth ready


In [11]:
from huggingface_hub import HfApi

REPO_URL = "https://github.com/pranjalparashar/FlowOS_v2.git"
ENV_URL = "https://praanjal-flowos-v2.hf.space"
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_MODEL_REPO = "praanjal/flowos-sft-ranked"
GRPO_RUNS_REPO = "praanjal/flowos-grpo-runs"
HF_FLAVOR = "l4x1"

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=SFT_MODEL_REPO, repo_type="model", exist_ok=True)
api.create_repo(repo_id=GRPO_RUNS_REPO, repo_type="dataset", exist_ok=True)
print("Output repos are ready")

Output repos are ready


### 14:44 - final training

In [13]:
from huggingface_hub import run_job

job = run_job(
    image="python:3.11",
    flavor="l40sx1",
    timeout="14h",
    command=[
        "bash",
        "-lc",
        f"""
        set -e
        export HF_TOKEN="{HF_TOKEN}"
        export HUGGING_FACE_HUB_TOKEN="{HF_TOKEN}"

        git clone https://github.com/pranjalparashar/FlowOS_v2.git repo
        cd repo

        pip install -U pip
        pip install -e '.[train]'
        pip install matplotlib

        python - <<'PY'
import os
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF login ok")
PY

        python collect_traces.py \
          --env-url https://praanjal-flowos-v2.hf.space \
          --task-scope simulate_csv_report_workflow \
          --dataset-size 112 \
          --max-turns 12 \
          --output-dir outputs/sft_traces_mistral_final

        CUDA_VISIBLE_DEVICES=0 python train_sft.py \
          --dataset-path outputs/sft_traces_mistral_final/traces.jsonl \
          --model-id mistralai/Mistral-7B-Instruct-v0.3 \
          --output-dir outputs/flowos-sft-mistral-final \
          --learning-rate 1e-6 \
          --num-epochs 6 \
          --per-device-train-batch-size 1 \
          --gradient-accumulation-steps 4 \
          --max-seq-length 1024 \
          --load-in-4bit \
          --gradient-checkpointing

        RUN_DIR=$(python - <<'PY'
from pathlib import Path
runs = sorted(Path("outputs").glob("flowos-sft-mistral-final-*"))
if not runs:
    raise SystemExit("No saved SFT run folder found.")
print(runs[-1])
PY
)
        echo "Using run dir: $RUN_DIR"

        python plot_metrics.py \
          --train-metrics "$RUN_DIR/training_metrics.csv" \
          --output-dir outputs/plots_mistral_final_train

        python - <<'PY'
import os
from pathlib import Path
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])

model_repo = "praanjal/flowos-sft-mistral-final"
artifact_repo = "praanjal/flowos-final-artifacts"

runs = sorted(Path("outputs").glob("flowos-sft-mistral-final-*"))
if not runs:
    raise SystemExit("No saved SFT run folder found for upload.")
run_dir = runs[-1]
run_name = run_dir.name

api.create_repo(repo_id=model_repo, repo_type="model", exist_ok=True)
api.upload_folder(
    repo_id=model_repo,
    repo_type="model",
    folder_path=str(run_dir),
)
print("Uploaded model to", model_repo)

api.create_repo(repo_id=artifact_repo, repo_type="dataset", exist_ok=True)
api.upload_folder(
    repo_id=artifact_repo,
    repo_type="dataset",
    folder_path=str(run_dir),
    path_in_repo=f"{{run_name}}/train_run",
)
api.upload_folder(
    repo_id=artifact_repo,
    repo_type="dataset",
    folder_path="outputs/plots_mistral_final_train",
    path_in_repo=f"{{run_name}}/train_plots",
)
print("Uploaded training artifacts and plots to", artifact_repo, "under", run_name)
PY
        """
    ],
)

print("Job ID:", job.id)
print("Job URL:", job.url)


Job ID: 69edd7c7d2c8bd8662bcfae2
Job URL: https://huggingface.co/jobs/praanjal/69edd7c7d2c8bd8662bcfae2


### 12:15-eval

In [41]:
from huggingface_hub import run_job

job = run_job(
    image="python:3.11",
    flavor="l40sx1",
    timeout="4h",
    command=[
        "bash",
        "-lc",
        f"""
        set -e
        export HF_TOKEN="{HF_TOKEN}"
        export HUGGING_FACE_HUB_TOKEN="{HF_TOKEN}"

        git clone https://github.com/pranjalparashar/FlowOS_v2.git repo
        cd repo

        pip install -U pip
        pip install -e '.[train]'
        pip install matplotlib

        python - <<'PY'
import os
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("HF login ok")
PY

        python eval.py \
          --model-id mistralai/Mistral-7B-Instruct-v0.3 \
          --checkpoint-path praanjal/flowos-sft-mistral-og \
          --env-url https://praanjal-flowos-v2.hf.space \
          --task-scope simulate_csv_report_workflow \
          --episodes 7 \
          --max-turns 12 \
          --disable-fallback \
          --debug-actions \
          --results-dir outputs/eval_compare_mistral_og_nofallback

        python plot_metrics.py \
          --eval-metrics outputs/eval_compare_mistral_og_nofallback/eval_episode_metrics.csv \
          --output-dir outputs/plots_mistral_og_nofallback

        python - <<'PY'
import os
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
repo_id = "praanjal/flowos-eval-artifacts"
api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

api.upload_folder(
    repo_id=repo_id,
    repo_type="dataset",
    folder_path="outputs/eval_compare_mistral_og_nofallback",
    path_in_repo="eval_compare_mistral_og_nofallback",
)

api.upload_folder(
    repo_id=repo_id,
    repo_type="dataset",
    folder_path="outputs/plots_mistral_og_nofallback",
    path_in_repo="plots_mistral_og_nofallback",
)

print("Uploaded artifacts to", repo_id)
PY
        """
    ],
)

print("Job ID:", job.id)
print("Job URL:", job.url)

Job ID: 69edc902d70108f37acdfe1f
Job URL: https://huggingface.co/jobs/praanjal/69edc902d70108f37acdfe1f
